# 🔍 AMRBART Vietnamese AMR Parser — Diverse Beam Search Inference

Inference notebook sử dụng **Diverse Beam Search** + **min_new_tokens** chống thin graph:
- `diversity_penalty`: chia beams thành nhóm, mỗi nhóm explore hướng khác nhau
- `min_new_tokens=30`: bắt buộc model sinh đủ tokens, tránh early EOS dẫn đến thin graph

```
Training:        Multinomial Sampling (T=0.8)    → grpo_trainer.py
Eval/Checkpoint: Beam Search (beams=5)           → main.py
Final Inference: Diverse Beam + min_new_tokens   ← notebook này
```

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR  = '/content/Enhance-AMR-BART_RL'
FINE_TUNE = f'{REPO_DIR}/fine-tune'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/Phucgiacat/Enhance-AMR-BART_RL.git {REPO_DIR}

os.chdir(FINE_TUNE)
print(f'CWD: {os.getcwd()}')

In [ ]:
!pip install -q penman smatch

## 2. Config

In [ ]:
import json

OUTPUT_DIR = '/content/drive/MyDrive/output_grpo/AMRBART-GRPO_v2'

# Auto-detect best checkpoint
with open(f'{OUTPUT_DIR}/trainer_state.json') as f:
    state = json.load(f)
CHECKPOINT = state['best_model_checkpoint']

# Hoặc chỉ định cụ thể:
# CHECKPOINT = f'{OUTPUT_DIR}/checkpoint-1420'

print(f'Checkpoint:  {CHECKPOINT}')
print(f'eval_smatch: {state["best_metric"]}')
print(f'Epoch:       {state["epoch"]:.2f}')

# ── Parameters ───────────────────────────────────────────────────────────────
NUM_BEAMS         = 10    # chia hết cho NUM_BEAM_GROUPS
NUM_BEAM_GROUPS   = 5
DIVERSITY_PENALTY = 0.8
MIN_NEW_TOKENS    = 30    # ← chống thin graph: bắt buộc sinh ít nhất 30 tokens
MAX_NEW_TOKENS    = 400
BATCH_SIZE        = 4     # giảm xuống 2 nếu OOM

## 3. Load Model & Tokenizer

In [ ]:
import sys, torch, penman
sys.path.insert(0, FINE_TUNE)

from transformers import BartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer

print(f'Loading tokenizer from: {CHECKPOINT}')
tokenizer = AMRBartTokenizer.from_pretrained(CHECKPOINT, use_fast=False)

print(f'Loading model...')
model = BartForConditionalGeneration.from_pretrained(CHECKPOINT)
model.eval().cuda()

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\nModel: {params:.0f}M params')
print(f'GPU:   {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## 4. Inference Functions

In [ ]:
from tqdm.notebook import tqdm

def decode_to_amr(token_ids, tokenizer):
    DUMMY = '(z / amr-empty)'
    seq = list(map(int, token_ids))
    if not seq:
        return DUMMY
    seq[0] = tokenizer.bos_token_id
    clean = []
    for tok in seq:
        if tok == tokenizer.pad_token_id:
            break
        mapped = tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
        clean.append(mapped)
        if mapped == tokenizer.eos_token_id:
            break
    try:
        graph, _, _ = tokenizer.decode_amr(clean, restore_name_ops=False)
        enc = penman.encode(graph)
        return enc if enc.strip() not in ('()', '') else DUMMY
    except:
        return DUMMY


def parse_diverse_beam(sentences, verbose=True):
    results = []
    it = tqdm(range(0, len(sentences), BATCH_SIZE)) if verbose \
         else range(0, len(sentences), BATCH_SIZE)

    for i in it:
        batch = sentences[i: i + BATCH_SIZE]
        inputs = tokenizer(
            batch, return_tensors='pt',
            padding=True, truncation=True, max_length=512,
        ).to('cuda')

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                # ── Diverse Beam Search ──
                num_beams=NUM_BEAMS,
                num_beam_groups=NUM_BEAM_GROUPS,
                diversity_penalty=DIVERSITY_PENALTY,
                # ── Chống thin graph ──
                min_new_tokens=MIN_NEW_TOKENS,
                # ────────────────────────
                max_new_tokens=MAX_NEW_TOKENS,
                forced_bos_token_id=tokenizer.amr_bos_token_id,
                early_stopping=True,
            )

        for seq in outputs:
            results.append(decode_to_amr(seq.tolist(), tokenizer))

    return results


def show(sentence, amr, idx=0):
    print(f'\n{"─"*65}')
    print(f'# ::id {idx}')
    print(f'# ::annotator diverse-beam-amr')
    print(f'# ::snt {sentence}')
    print(amr)

print('Ready ✅')

## 5. Test — Single Sentences

In [ ]:
test_cases = [
    "Hãy cùng lắng_nghe bài hát mà Coca-Cola đã tạo ra, Lá cờ bay được biểu_diễn bởi nghệ_sĩ hip hop người Somalia",
    "dẫu_sao cứ hãy cứ khâm_phục ta!",
    "Anh ấy không biết rằng cô gái đó đã về quê từ hôm qua",
    "Chính_phủ Việt_Nam đã phê_duyệt dự_án đầu_tư trị_giá 500 triệu đô",
]

results = parse_diverse_beam(test_cases)

for i, (sent, amr) in enumerate(zip(test_cases, results)):
    show(sent, amr, idx=i)

## 6. Batch Inference từ File

In [ ]:
import time

INPUT_FILE  = '/content/drive/MyDrive/data/test_sentences.txt'   # mỗi dòng 1 câu
OUTPUT_FILE = '/content/drive/MyDrive/output_grpo/inference_diverse_beam.txt'

with open(INPUT_FILE, encoding='utf-8') as f:
    sentences = [l.strip() for l in f if l.strip()]
print(f'Loaded {len(sentences)} sentences')

t0 = time.time()
all_results = parse_diverse_beam(sentences)
elapsed = time.time() - t0

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for i, (sent, amr) in enumerate(zip(sentences, all_results)):
        f.write(f'# ::id {i}\n# ::annotator diverse-beam-amr\n# ::snt {sent}\n{amr}\n\n')

empty = sum(1 for r in all_results if 'amr-empty' in r)
thin  = sum(1 for r in all_results if 'amr-unknown' in r and r.count(':') <= 2)
print(f'\nDone in {elapsed:.1f}s ({elapsed/len(sentences):.2f}s/sent)')
print(f'amr-empty: {empty}/{len(all_results)} ({100*empty/len(all_results):.1f}%)')
print(f'thin+unk:  {thin}/{len(all_results)}  ({100*thin/len(all_results):.1f}%)')
print(f'Output: {OUTPUT_FILE}')

## 7. Eval Smatch (nếu có gold)

In [ ]:
GOLD_FILE = '/content/drive/MyDrive/data/test_gold.txt'

if os.path.exists(GOLD_FILE):
    from common.utils import calculate_smatch
    score = calculate_smatch(GOLD_FILE, OUTPUT_FILE)
    print(f'📊 Smatch (Diverse Beam, min_new_tokens={MIN_NEW_TOKENS}): F1 = {score["smatch"]:.4f}')
else:
    print(f'Gold file không tồn tại: {GOLD_FILE}')

## 8. Shell Script (alternative)

In [ ]:
%%bash
source /content/AMRBART/.venv/bin/activate

bash /content/Enhance-AMR-BART_RL/fine-tune/inference-diverse-beam.sh \
    "/content/drive/MyDrive/output_grpo/AMRBART-GRPO_v2/checkpoint-1420" \
    --sentence "Hãy cùng lắng_nghe bài hát mà Coca-Cola đã tạo ra"